# autosi デモ — Automatic Selective Inference

autosi は、特徴選択などのアルゴリズムを `asi` の配列演算で記述するだけで、
アルゴリズム内の全ての比較・分岐（= 選択イベント）を自動追跡し、
妥当な selective p-value を計算するフレームワーク。

使い方は 3 ステップ:
1. データを `asi.array` でラップする
2. 選択アルゴリズムを numpy ライクな `asi` 演算で書き、検定したい方向 `eta` を返す
3. `asi.inference(eta, prob_vec, var, algorithm)` を呼ぶ

各デモとも「アルゴリズムが特徴集合を選択し、選択された集合での 回帰係数を検定する」
という標準的な設定。（ノートブック全体の実行時間は約 20 秒）

## デモ 1: Marginal Screening

相関の絶対値が最も大きい特徴を 1 つ選択し、その回帰係数を検定する。
選択イベントは $z$ の 1 次不等式になる。

In [1]:
import numpy as np

# 帰無仮説の下でデータを生成 (beta = 0)
rng = np.random.default_rng(0)
n, p = 100, 10
X_np = rng.normal(size=(n, p))
y_np = rng.normal(size=n)

In [2]:
import autosi as asi

# データをラップ
X = asi.array(X_np)
y = asi.array(y_np)


def marginal_screening(y):
    corr = X.T @ y                     # 各特徴との相関
    best = asi.argmax(asi.abs(corr))   # 最も相関の強い特徴を選択
    x_M = X.T[best]
    eta = x_M / (x_M @ x_M)
    return eta            # 選択された特徴の回帰係数のコントラスト

In [3]:
eta = marginal_screening(y)
result = asi.inference(eta=eta, prob_vec=y, var=1.0, algorithm=marginal_screening)

print(f"selective p = {result.p_value:.4f}, naive p = {result.naive_p_value():.4f}")

selective p = 0.2315, naive p = 0.0403


## デモ 2: Lasso（座標降下法）

目的関数は $\frac{1}{2n}\|y - X\beta\|^2 + \lambda\|\beta\|_1$。
ソフト閾値処理の `if / elif / else` 分岐がそのまま選択イベントとして自動追跡される。
非ゼロ係数の特徴集合を選択し、その集合での回帰係数を検定する。

In [4]:
# lasso デモ用のデータとハイパーパラメータ
rng = np.random.default_rng(0)
n, p = 100, 10
X_np = rng.normal(size=(n, p))
y_np = rng.normal(size=n)

X = asi.array(X_np)
y = asi.array(y_np)

lam = 0.1
max_iter = 100
tol = 1e-4

In [5]:
def lasso(y):
    col_sq_mean = asi.sum(X**2, axis=0) / n
    coef = asi.array(np.zeros(p))
    for _ in range(max_iter):
        coef_old = coef.copy()
        for j in range(p):
            # 部分残差とソフト閾値処理による更新
            r = y - X @ coef + X[:, j] * coef[j]
            rho = X[:, j] @ r / n
            if rho > lam:
                coef[j] = (rho - lam) / col_sq_mean[j]
            elif rho < -lam:
                coef[j] = (rho + lam) / col_sq_mean[j]
            else:
                coef[j] = asi.array(0.0)
        if asi.sum(asi.abs(coef - coef_old)) < tol:
            break

    active = coef != 0
    X_M = X.T[active]
    eta = (X_M @ X_M.T).inv() @ X_M     # 選択された特徴集合上の回帰係数のコントラスト
    return eta[0]                       # 特徴集合の中の最初の特徴を検定

In [6]:
eta = lasso(y)
result = asi.inference(eta=eta, prob_vec=y, var=1.0, algorithm=lasso)

print(f"selective p = {result.p_value:.4f}, naive p = {result.naive_p_value():.4f}")

selective p = 0.2349, naive p = 0.0460


## デモ 3: クロスバリデーションによる正則化パラメータ選択（8 次多項不等式）

実務で標準的な「$K$-fold CV の $R^2$ で lasso の $\lambda$ を選び、
選ばれた $\lambda$ での lasso が選択した特徴集合の係数を検定する」設定。
CV スコアは $z$ の **4 次 / 4 次の有理式** になるため、$\lambda$ 候補間の比較から
8 次の多項不等式を解く。1・2 次の不等式しか扱えない既存手法の範囲を超える
選択イベントも、そのまま記述するだけでよい。

In [7]:
rng = np.random.default_rng(0)
n, p = 50, 5
K = 2
lambdas = [0.02, 0.1, 0.5]  # λ のグリッド
max_iter = 30
tol = 1e-3

# シグナルありのデータ
X_np = rng.normal(size=(n, p))
beta = np.array([0.5, 0.5, 0.0, 0.0, 0.0])  # 特徴 0, 1 が真の信号を持つ
y_np = X_np @ beta + rng.normal(size=n)
fold_indices = np.array_split(rng.permutation(n), K)

X = asi.array(X_np)
y = asi.array(y_np)

In [8]:
def cv_lasso(y):
    def solve(X_tr, y_tr, lam, n_tr):
        # lasso (座標降下法)
        col_sq_mean = asi.sum(X_tr**2, axis=0) / n_tr
        coef = asi.array(np.zeros(p))
        for _ in range(max_iter):
            coef_old = coef.copy()
            for j in range(p):
                r = y_tr - X_tr @ coef + X_tr[:, j] * coef[j]
                rho = X_tr[:, j] @ r / n_tr
                if rho > lam:
                    coef[j] = (rho - lam) / col_sq_mean[j]
                elif rho < -lam:
                    coef[j] = (rho + lam) / col_sq_mean[j]
                else:
                    coef[j] = asi.array(0.0)
            if asi.sum(asi.abs(coef - coef_old)) < tol:
                break
        return coef

    # K-fold CV で各 λ のスコア (R² の平均) を計算
    scores = []
    for lam in lambdas:
        score = asi.array(0.0)
        for fold in range(K):
            tr = np.concatenate([fold_indices[j] for j in range(K) if j != fold])
            va = fold_indices[fold]
            coef = solve(X[tr], y[tr], lam, len(tr))
            ss_res = asi.sum((y[va] - X[va] @ coef) ** 2)
            ss_tot = asi.sum((y[va] - asi.sum(y[va]) / len(va)) ** 2)
            score = score + 1.0 - ss_res / ss_tot
        scores.append(score / K)

    # CV スコア最大の λ を選び、全データで lasso を実行
    best = asi.argmax(asi.stack(scores))
    coef = solve(X, y, lambdas[best], n)

    X_M = X.T[coef != 0]
    eta = (X_M @ X_M.T).inv() @ X_M             # 選択された特徴集合上の回帰係数のコントラスト
    return eta[0]                               # 特徴集合の中の最初の特徴を検定

In [9]:
eta = cv_lasso(y)
result = asi.inference(eta=eta, prob_vec=y, var=1.0, algorithm=cv_lasso)

print(f"selective p = {result.p_value:.4f}")

selective p = 0.0348
